# Restriction site scan

讀取 06 組好的 `outputs/06_whole_sequence.csv`，對每條 `full_sequence` 掃描限制酶切位，
再用 06 記下的 segment offsets 判斷每個切位落在構築的哪一段。

- 同時掃正股與反股（非回文的 **Eco31I (GGTCTC)** 才需要，回文酶兩股相同）。
- BG5 / BG3 / RE1 / RE2 裡的切位是設計好的，計入 `sites_expected`。
- 落在 promoter 或 barcode 內、或跨越接縫的，計入 `sites_unexpected` — 那才是要處理的。

酶表與 `find_re_sites()` 都來自 `_paths`，這裡不再重複定義。

**輸入** `outputs/06_whole_sequence.csv`
**輸出** `outputs/07_re_scan_report.csv`

In [1]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


PROJECT_ROOT : C:\project\Whole-model
DATA_DIR     : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data
RELEASE_OUT  : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs


## Standardised report for the whole library

Uses the segment offsets recorded by 06 to say *where* each site sits.
A site inside BG5 / RE1 / RE2 / BG3 is by design; a site inside the promoter
or barcode, or spanning a junction, is the thing worth acting on.

Writes `outputs/07_re_scan_report.csv`.

In [2]:
# === 07_re_scan_report.csv: per-candidate sites attributed to a segment ===
import pandas as pd

lib = pd.read_csv(require(RELEASE_OUT / "06_whole_sequence.csv", "assembled library"),
                  low_memory=False)
print("candidates:", len(lib))

RE1 = lib["re1_seq"].iloc[0]
RE2 = lib["re2_seq"].iloc[0]
print(f"construct RE1={RE1}  RE2={RE2}")


def segments(row):
    """(name, start, end) for one construct, end-exclusive."""
    return [
        ("BG5", 0, row["bg5_end"]),
        ("promoter", row["promoter_start"], row["promoter_end"]),
        ("RE1", row["re1_start"], row["re2_start"]),
        ("RE2", row["re2_start"], row["barcode_start"]),
        ("barcode", row["barcode_start"], row["bg3_start"]),
        ("BG3", row["bg3_start"], row["bg3_end"]),
    ]


def locate(pos, length, segs):
    """Which segment(s) a site at [pos, pos+length) touches."""
    return "+".join(n for n, a, b in segs if pos < b and pos + length > a)


# A site is "expected" only when it lies wholly inside one constant segment:
# the BsaI sites built into BG5/BG3, and RE1/RE2 themselves.
EXPECTED = {"BG5", "BG3", "RE1", "RE2"}

rows = []
for row in lib.itertuples(index=False):
    r = row._asdict()
    segs = segments(r)
    seq = r["full_sequence"]
    rec = {
        "candidate_id": r["candidate_id"],
        "source": r["source"],
        "full_length": r["full_length"],
    }
    n_expected = n_extra = 0
    extra_where = []
    for enz, site in RE_SITES.items():
        hits = find_re_sites(seq, site)
        extra = 0
        for p in hits:
            where = locate(p, len(site), segs)
            if where in EXPECTED:
                n_expected += 1
            else:
                extra += 1
                extra_where.append(f"{enz}@{p}({where})")
        rec[enz] = extra
        n_extra += extra
    rec["sites_expected"] = n_expected
    rec["sites_unexpected"] = n_extra
    rec["clean"] = n_extra == 0
    rec["unexpected_detail"] = "; ".join(extra_where)
    rows.append(rec)

report = pd.DataFrame(rows)
dest = RELEASE_OUT / "07_re_scan_report.csv"
report.to_csv(dest, index=False)
print(f"\nwrote {dest}  {report.shape}")


candidates: 33999
construct RE1=GTCGAC  RE2=TCTAGA



wrote C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\07_re_scan_report.csv  (33999, 26)


In [3]:
# === Summary: what actually needs a decision ===
enz_cols = [e for e in RE_SITES if e in report.columns]

print(f"clean constructs: {int(report['clean'].sum())} / {len(report)} "
      f"({report['clean'].mean():.1%})\n")

print("clean rate by source:")
print(report.groupby("source")["clean"].agg(["size", "sum", "mean"]).to_string())

per_enzyme = (
    pd.DataFrame({
        "enzyme": enz_cols,
        "site": [RE_SITES[e] for e in enz_cols],
        "n_constructs": [(report[e] > 0).sum() for e in enz_cols],
        "n_sites": [report[e].sum() for e in enz_cols],
    })
    .sort_values("n_constructs", ascending=False)
    .reset_index(drop=True)
)
print("\nunexpected sites per enzyme (outside BG5/BG3/RE1/RE2):")
print(per_enzyme[per_enzyme["n_constructs"] > 0].to_string(index=False))

zero = per_enzyme[per_enzyme["n_constructs"] == 0]["enzyme"].tolist()
print(f"\nenzymes with zero unexpected sites library-wide: {zero}")
print("These are the safe candidates if RE1/RE2 need to be swapped.")

print("\nexample offenders:")
bad = report[~report["clean"]]
if len(bad):
    print(bad[["candidate_id", "source", "sites_unexpected", "unexpected_detail"]].head(10).to_string(index=False))
else:
    print("none")


clean constructs: 20727 / 33999 (61.0%)

clean rate by source:
            size   sum      mean
source                          
assembled  15625  6376  0.408064
native      5514  4557  0.826442
phage      12860  9794  0.761586

unexpected sites per enzyme (outside BG5/BG3/RE1/RE2):
 enzyme     site  n_constructs  n_sites
   XbaI   TCTAGA          4044     4048
  EcoRI   GAATTC          3410     3410
   NheI   GCTAGC          2531     2607
   SacI   GAGCTC          1963     1964
   VspI   ATTAAT          1197     1261
   NdeI   CATATG          1086     1101
HindIII   AAGCTT           501      521
   XhoI   CTCGAG           346      348
   NcoI   CCATGG           306      306
   BcuI   ACTAGT           254      255
 Eco31I   GGTCTC           202      203
  BglII   AGATCT           199      201
   PstI   CTGCAG           184      186
   MluI   ACGCGT           170      171
   SmaI   CCCGGG           158      158
   SalI   GTCGAC            97       97
  BamHI   GGATCC            67      